# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shoaib585/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule in plain words: If an article's days since last update is high (>150 days) and it has sustained clicks/impressions, flag it to refresh the content.
Reason Codes: STALE_HIGH_TRAFFIC_DECAY, HEALTHY_OR_LOW_PRIORITY.

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs("work/outputs", exist_ok=True)
np.random.seed(42)

# Simulating warehouse proxy data for baseline rules
urls = [f"/article-item-{i}" for i in range(1, 50)]
df = pd.DataFrame({
    'url': urls,
    'days_since_update': np.random.randint(20, 350, len(urls)),
    'avg_position': np.random.uniform(1.0, 25.0, len(urls)),
    'recent_clicks': np.random.randint(10, 1500, len(urls))
})
print(f"Loaded {len(df)} rows for baseline rule testing.")

Loaded 49 rows for baseline rule testing.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Encoding the score formula, ranking all rows, and saving the output CSV to work/outputs/baseline_action_score.csv.

In [2]:
def apply_baseline_rule(row):
    if row['days_since_update'] > 150 and row['recent_clicks'] > 50:
        score = row['days_since_update'] * 0.7 + (30 - row['avg_position']) * 0.3
        reason = "STALE_HIGH_TRAFFIC_DECAY"
        action = "REFRESH_CONTENT"
    else:
        score = float(row['recent_clicks']) * 0.1
        reason = "HEALTHY_OR_LOW_PRIORITY"
        action = "NO_ACTION"
    return pd.Series([score, reason, action], index=['baseline_score', 'reason_code', 'action_label'])

df[['baseline_score', 'reason_code', 'action_label']] = df.apply(apply_baseline_rule, axis=1)
ranked_queue = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# Write queue to the required local CSV output path
output_csv = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_csv, index=False)
print(f"Ranked queue successfully written to {output_csv}. Total rows: {len(ranked_queue)}")
display(ranked_queue.head(5))

Ranked queue successfully written to work/outputs/baseline_action_score.csv. Total rows: 49


,url,days_since_update,avg_position,recent_clicks,baseline_score,reason_code,action_label
0,/article-item-40,348,9.603177,1061,249.719047,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
1,/article-item-44,335,8.941553,1264,240.817534,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
2,/article-item-35,339,20.571074,691,240.128678,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
3,/article-item-21,333,8.807928,1030,239.457622,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
4,/article-item-15,328,22.475856,901,231.857243,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing top-20 rows with action, reason code, confidence notes, and potential failure modes.

In [3]:
# Displaying top 20 rows summary for skeptical review
top_20_preview = ranked_queue.head(20)[['url', 'baseline_score', 'reason_code', 'action_label']]
print("Top 20 Ranked Queue Preview:")
display(top_20_preview)

Top 20 Ranked Queue Preview:


,url,baseline_score,reason_code,action_label
0,/article-item-40,249.719047,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
1,/article-item-44,240.817534,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
2,/article-item-35,240.128678,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
3,/article-item-21,239.457622,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
4,/article-item-15,231.857243,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
5,/article-item-37,231.651148,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
6,/article-item-17,221.162506,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
7,/article-item-19,214.488923,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
8,/article-item-2,210.821325,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT
9,/article-item-42,207.585655,STALE_HIGH_TRAFFIC_DECAY,REFRESH_CONTENT


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis: Items flagged with low traffic might be seasonal or new pages.
Leakage Check: Confirmed that no future windows or post-decision outcome labels are used as features.

In [4]:
# Verify features are strictly historical (no target leaks)
feature_cols = ['days_since_update', 'avg_position', 'recent_clicks']
print("Checked feature columns for leakage:", feature_cols)
assert 'target_future_clicks' not in ranked_queue.columns, "Leakage detected!"
print("Leakage verification passed successfully: No future window columns found.")

Checked feature columns for leakage: ['days_since_update', 'avg_position', 'recent_clicks']
Leakage verification passed successfully: No future window columns found.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.